In [0]:
df_reviews = spark.table("ecommerce_dev.bronze.order_reviews")

df_reviews.printSchema()
display(df_reviews.limit(10))

print(f"Total rows: {df_reviews.count()}")

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_rescued_data,_ingested_at,_source_file,_source_table
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews


Total rows: 99224


In [0]:
from pyspark.sql.functions import *

# 1. Duplicate review_id check
dupes = df_reviews.groupBy("review_id").count().filter("count > 1")
print(f"Duplicate review_ids: {dupes.count()}")

# 2. Null rates on key columns
df_reviews.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ["review_id", "order_id", "review_score", "review_comment_title", 
      "review_comment_message", "review_creation_date", "review_answer_timestamp"]]
).display()

# 3. review_score value range (sanity check for cast to int, e.g. any non-numeric junk)
df_reviews.groupBy("review_score").count().orderBy("review_score").display()

# 4. FK integrity against silver.orders
orphans = df_reviews.join(
    spark.table("ecommerce_dev.silver.orders").select("order_id"),
    on="order_id", how="left_anti"
)
print(f"Orphaned order_ids (no match in silver.orders): {orphans.count()}")

# 5. Rescued data check
print(f"Rows with _rescued_data populated: {df_reviews.filter(col('_rescued_data').isNotNull()).count()}")

Duplicate review_ids: 789


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,0,0,87656,58247,0,0


review_score,count
1,11424
2,3151
3,8179
4,19142
5,57328


Orphaned order_ids (no match in silver.orders): 0
Rows with _rescued_data populated: 0


In [0]:
# Refresh df_reviews to point at the corrected Bronze table
df_reviews = spark.table("ecommerce_dev.bronze.order_reviews")

# Pull a sample of duplicate review_ids to see the pattern
dupe_ids = (df_reviews.groupBy("review_id").count()
            .filter("count > 1")
            .select("review_id"))

dupe_detail = (df_reviews.join(dupe_ids, on="review_id")
               .orderBy("review_id"))

display(dupe_detail.limit(30))

# Check: are duplicates exact full-row duplicates, or same review_id with different order_id/content?
exact_dupes = df_reviews.groupBy(df_reviews.columns).count().filter("count > 1")
print(f"Exact full-row duplicate groups: {exact_dupes.count()}")

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_rescued_data,_ingested_at,_source_file,_source_table
00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,null,"O cartucho ""original HP"" 60XL não é reconhecido pela impressora, consequentemente não funcionou. Além de ter chegado com atraso de mais de 15 dias do previsto. Preciso que seja trocado.",2018-03-07 00:00:00,2018-03-20 18:08:23,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,null,null,2017-09-21 00:00:00,2017-09-26 03:27:47,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,null,null,2017-09-21 00:00:00,2017-09-26 03:27:47,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,null,Produto entregue dentro de embalagem do fornecedor sem os parafusos de fixação das partes.,2018-03-07 00:00:00,2018-03-08 03:00:53,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,null,null,2018-03-02 00:00:00,2018-03-05 01:43:30,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,null,null,2018-03-02 00:00:00,2018-03-05 01:43:30,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09 00:00:00,2017-09-13 09:52:44,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews
0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,null,"O pedido consta de 2 produtos e até agora recebi apenas 1 produto, e o que me preocupa é que o status aparece como entregue. Solicito providências",2017-09-09 00:00:00,2017-09-13 09:52:44,null,2026-08-08T19:16:29.768Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv,order_reviews


Exact full-row duplicate groups: 0


In [0]:
from pyspark.sql.functions import countDistinct

content_variance_check = (df_reviews.groupBy("review_id")
    .agg(
        countDistinct("review_score").alias("distinct_scores"),
        countDistinct("review_comment_message").alias("distinct_messages"),
        countDistinct("review_answer_timestamp").alias("distinct_answer_ts"),
        count("*").alias("row_count")
    )
    .filter("row_count > 1")
)

# Any review_id where the non-order_id fields actually differ?
inconsistent = content_variance_check.filter(
    "distinct_scores > 1 OR distinct_messages > 1 OR distinct_answer_ts > 1"
)
print(f"review_ids with inconsistent content across duplicate rows: {inconsistent.count()}")
display(inconsistent)

review_ids with inconsistent content across duplicate rows: 0


review_id,distinct_scores,distinct_messages,distinct_answer_ts,row_count


In [0]:
print(f"Null review_scores: {df_reviews.filter(col('review_score').isNull()).count()}")

Null review_scores: 0


In [0]:
from pyspark.sql.functions import *

# --- Load from Bronze ---
df_reviews = spark.table("ecommerce_dev.bronze.order_reviews")

# --- Transform ---
df_silver_reviews = (df_reviews
    .withColumn("review_score", col("review_score").cast("int"))
    .withColumn("review_creation_date", col("review_creation_date").cast("timestamp"))
    .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast("timestamp"))
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
    .drop("_rescued_data", "_source_file", "_source_table")
)

# --- Write to Silver ---
(df_silver_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_dev.silver.order_reviews")
)


In [0]:

# --- Enforce constraints ---
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_reviews
    ALTER COLUMN review_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_reviews
    ALTER COLUMN order_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_reviews
    ADD CONSTRAINT pk_order_reviews PRIMARY KEY (review_id, order_id)
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_reviews
    ADD CONSTRAINT fk_order_reviews_order_id FOREIGN KEY (order_id)
    REFERENCES ecommerce_dev.silver.orders (order_id)
""")


DataFrame[]

In [0]:

# --- Table comment ---
spark.sql("""
    COMMENT ON TABLE ecommerce_dev.silver.order_reviews IS
    'Customer reviews for orders. Note: review_id is NOT unique alone — a single 
    customer review can map to multiple order_ids when a checkout splits across 
    sellers (Olist marketplace behavior). Primary key is the composite (review_id, order_id). 
    review_comment_title/review_comment_message are legitimately null when no 
    written comment was left. review_score cast from string to int; date columns 
    cast from string to timestamp.'
""")

DataFrame[]